In [38]:
import subprocess
import pandas as pd
import numpy as np
import json
import argparse
from pathlib import Path
import shutil
from typing import Dict, List, Tuple
import sys

# Add GenNet_utils to path
sys.path.insert(0, str(Path("..").resolve()))

# Import simulation utilities
from GenNet_utils.simulation_utils import (
    create_effect_file_from_snps,
    calculate_grs_from_effects,
    apply_liability_threshold_model,
    create_gennet_subject_file,
    simulate_pathway_phenotype
)

In [39]:
n_samples = 3202
vcf_file = Path("../data/raw/geno_phased_fixed.vcf.gz")
output_dir = Path("../data/processed/simulationx")

## Genotyping Data Conversion

Converts VCF genotype file to PLINK format, and randomly samples n rows from this file. 

In [3]:
temp_prefix = output_dir / "temp_all"
cmd = f"""
plink2 --vcf {vcf_file} \
        --make-bed \
        --out {temp_prefix}
"""
subprocess.run(cmd, shell=True, check=True)

# read fam file to get sample info 
fam = pd.read_csv(f"{temp_prefix}.fam", sep=r'\s+', header=None,
                    names=['fid', 'iid', 'father', 'mother', 'sex', 'pheno'])

# select n random samples
n_available = len(fam)
n_to_select = min(n_samples, n_available)
selected_indices = np.random.choice(n_available, n_to_select, replace=False)
selected_samples = fam.iloc[selected_indices]

# Create keep file with FID and IID columns
samples_file = output_dir / "samples_keep.txt"
selected_samples[['fid', 'iid']].to_csv(samples_file, sep='\t', header=False, index=False)

# create plink clean directory
plink_clean_dir = output_dir / "plink_clean"
plink_clean_dir.mkdir(exist_ok=True)
plink_prefix = plink_clean_dir / "chr22_data"

cmd = f"""
plink2 --bfile {temp_prefix} \
        --keep {samples_file} \
        --make-bed \
        --maf 0.01 \
        --geno 0.05 \
        --out {plink_prefix}
"""
subprocess.run(cmd, shell=True, check=True)

# clean up temp files 
for ext in ['.bed', '.bim', '.fam', '.log']:
    temp_file = Path(f"{temp_prefix}{ext}")
    if temp_file.exists():
        temp_file.unlink()

plink_prefix = str(plink_prefix)

PLINK v2.0.0-a.7 M1 (20 Sep 2025)                   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to ../data/processed/simulationx/temp_all.log.
Options in effect:
  --make-bed
  --out ../data/processed/simulationx/temp_all
  --vcf ../data/raw/geno_phased_fixed.vcf.gz

Start time: Sun Jan  4 19:14:37 2026
16384 MiB RAM detected; reserving 8192 MiB for main workspace.
Using up to 10 threads (change this with --threads).
--vcf: 1070401 variants scanned.
--vcf: ../data/processed/simulationx/temp_all-temporary.pgen +
../data/processed/simulationx/temp_all-temporary.pvar.zst +
../data/processed/simulationx/temp_all-temporary.psam written.
3202 samples (0 females, 0 males, 3202 ambiguous; 3202 founders) loaded from
../data/processed/simulationx/temp_all-temporary.psam.
1070401 variants loaded from
../data/processed/simulationx/temp_all-temporary.pvar.zst.
Note: No phenotype data present.
Writing ../data/processed/simulatio

## LD Pruning

In [43]:
plink_pruned_dir = plink_clean_dir / "chr22_pruned"

cmd_prune = f"""
plink2 --bfile {plink_prefix} \
        --indep-pairwise 50 5 0.005 \
        --out {plink_pruned_dir}
"""
subprocess.run(cmd_prune, shell=True, check=True)


PLINK v2.0.0-a.7 M1 (20 Sep 2025)                   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to ../data/processed/simulationx/plink_clean/chr22_pruned.log.
Options in effect:
  --bfile ../data/processed/simulationx/plink_clean/chr22_pruned
  --indep-pairwise 50 5 0.005
  --out ../data/processed/simulationx/plink_clean/chr22_pruned

Start time: Tue Jan  6 12:20:53 2026
16384 MiB RAM detected; reserving 8192 MiB for main workspace.
Using up to 10 threads (change this with --threads).
3202 samples (0 females, 0 males, 3202 ambiguous; 3202 founders) loaded from
../data/processed/simulationx/plink_clean/chr22_pruned.fam.
39111 variants loaded from
../data/processed/simulationx/plink_clean/chr22_pruned.bim.
Note: No phenotype data present.
Calculating allele frequencies... done.
--indep-pairwise (1 compute thread): 5036845/39111 variants removed.
Variant lists written to
../data/processed/simulationx/plink_clean/chr22

CompletedProcess(args='\nplink2 --bfile ../data/processed/simulationx/plink_clean/chr22_pruned         --indep-pairwise 50 5 0.005         --out ../data/processed/simulationx/plink_clean/chr22_pruned\n', returncode=0)

In [44]:
with open(f'{plink_pruned_dir}.prune.in', 'r') as f: 
    print(len(f.readlines()))

2266


In [45]:
cmd_extract = f"""
plink2 --bfile {plink_prefix} \
       --extract {plink_pruned_dir}.prune.in \
       --make-bed \
       --out {plink_pruned_dir}
"""
subprocess.run(cmd_extract, shell=True, check=True)

plink_prefix = str(plink_pruned_dir)


PLINK v2.0.0-a.7 M1 (20 Sep 2025)                   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to ../data/processed/simulationx/plink_clean/chr22_pruned.log.
Options in effect:
  --bfile ../data/processed/simulationx/plink_clean/chr22_pruned
  --extract ../data/processed/simulationx/plink_clean/chr22_pruned.prune.in
  --make-bed
  --out ../data/processed/simulationx/plink_clean/chr22_pruned

Start time: Tue Jan  6 12:21:02 2026
16384 MiB RAM detected; reserving 8192 MiB for main workspace.
Using up to 10 threads (change this with --threads).
3202 samples (0 females, 0 males, 3202 ambiguous; 3202 founders) loaded from
../data/processed/simulationx/plink_clean/chr22_pruned.fam~.
39111 variants loaded from
../data/processed/simulationx/plink_clean/chr22_pruned.bim~.
Note: No phenotype data present.
--extract: 2266 variants remaining.
2266 variants remaining after main filters.
Writing ../data/processed/simulationx/pl

filenames.


## HDF5 File Conversion

In [46]:
convert_dir = output_dir / "gennet_convert_temp"
if convert_dir.exists():
    shutil.rmtree(convert_dir)
convert_dir.mkdir()

# copy only plink files to the directory 
base_name = Path(plink_prefix).name
for ext in ['.bed', '.bim', '.fam']:
    src = Path(f"{plink_prefix}{ext}")
    dst = convert_dir / f"{base_name}{ext}"
    shutil.copy2(src, dst)

h5_output_dir = output_dir / "h5_output"
if h5_output_dir.exists():
    shutil.rmtree(h5_output_dir)
h5_output_dir.mkdir()

gennet_script = Path("../GenNet.py")

cmd = f"""python {gennet_script} convert \
    -g {convert_dir} \
    -study_name {base_name} \
    -o {h5_output_dir}"""

subprocess.run(cmd, shell=True, check=True)

if convert_dir.exists():
    shutil.rmtree(convert_dir)

using ../data/processed/simulationx/h5_output/
Number of Individuals: 3202
Number of Probes 2266 in chr22_pruned.bim
Number of Probes 2266 converted
next 2266 SNPs, from 2266, need to convert 0
Time to read 2266 SNPs is 0.25109195709228516 s
Time to write 2266 SNPs is 0.013181924819946289 s
Number of individuals 3202 
  family individual  paternal  maternal  sex  label
0      0    HG00096         0         0    0     -9
1      0    HG00097         0         0    0     -9
2      0    HG00099         0         0    0     -9
3      0    HG00100         0         0    0     -9
4      0    HG00101         0         0    0     -9
Converted number of variants 2266
   CHR                ID  ...              allele1              allele2
0   22  22:10530385:GT:G  ... -4024417873775142397 -8088624436084440318
1   22   22:10561923:C:T  ...  7218356192587439341   710690809048561524
2   22   22:10562724:T:C  ...   710690809048561524  7218356192587439341
3   22   22:10571359:T:C  ...   71069080904856

  0%|          | 0/2 [00:00<?, ?it/s]

WARNING skipped step 4, all variants are used: using ../data/processed/simulationx/h5_output//chr22_pruned_step3_genotype_no_missing.h5
chuncksize = 3202
Completed chr22_pruned
You can delete all other h5 files if genotype.h5 is correct


100%|██████████| 2/2 [00:00<00:00, 34.00it/s]
